In [1]:
# ── Imports, chargement et merge ─────────────────────────────────────────────────────
# Chargement et merge des 4 fichiers CSV 
# 2021/2022/2023 → séparateur point-virgule (;)
# 2024 → séparateur virgule (,)
# Num_Acc forcé en string pour éviter les problèmes de jointure (float/int)
# Colonne annee ajoutée pour filtrer par année dans les analyses et PowerBI

import pandas as pd
import numpy as np

df_2021 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - lieux 2021.csv', sep=';', dtype={'Num_Acc': str})
df_2022 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - lieux 2022.csv', sep=';', dtype={'Num_Acc': str}, low_memory=False)
df_2023 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - lieux 2023.csv', sep=';', dtype={'Num_Acc': str}, low_memory=False)
df_2024 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - lieux 2024.csv', sep=',',  dtype={'Num_Acc': str})

df_2021['annee'] = 2021
df_2022['annee'] = 2022
df_2023['annee'] = 2023
df_2024['annee'] = 2024

lieux_df = pd.concat([df_2024, df_2023, df_2022, df_2021], ignore_index=True)
print(f"✅ Merge OK — {lieux_df.shape[0]:,} lignes x {lieux_df.shape[1]} colonnes")
lieux_df.head()

✅ Merge OK — 252,928 lignes x 19 colonnes


,Num_Acc,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma,annee
0,202400000001,3,D438,0,NaN,2,2,0,1,1,260,2,NaN,7.0,1,0,1,90,2024
1,202400000002,4,HOTEL DIEU (RUE DE L'),0,NaN,2,2,0,1,-1,-1,1,NaN,-1.0,9,0,1,30,2024
2,202400000002,4,POTERNE (RUE),0,NaN,1,1,0,1,-1,-1,1,NaN,-1.0,9,0,1,30,2024
3,202400000003,4,TILLEULS (ALLEE DES),0,NaN,2,2,0,1,-1,-1,1,NaN,-1.0,1,0,3,50,2024
4,202400000004,4,AUTHIE (NÂ° 106 PAIRS -115 IMPAIRS),0,NaN,2,4,0,1,-1,-1,1,NaN,-1.0,1,9,1,50,2024


In [2]:
# ── Suppression des colonnes inutiles ───────────────────────────
cols_to_drop = ['voie', 'v1', 'v2', 'pr', 'pr1',
                'lartpc', 'larrout', 'vosp', 'plan']

lieux_df = lieux_df.drop(columns=cols_to_drop)
print(f"✅ Colonnes supprimées — shape : {lieux_df.shape}")
print(f"Colonnes restantes : {list(lieux_df.columns)}")

✅ Colonnes supprimées — shape : (252928, 10)
Colonnes restantes : ['Num_Acc', 'catr', 'circ', 'nbv', 'prof', 'surf', 'infra', 'situ', 'vma', 'annee']


In [3]:
# ── Remplacement des -1 par NaN ─────────────────────────────────
cols_sentinel = ['catr', 'nbv', 'circ', 'prof', 'surf', 'infra', 'situ', 'vma']

for col in cols_sentinel:
    lieux_df[col] = lieux_df[col].replace(-1, np.nan)
    lieux_df[col] = lieux_df[col].replace("-1", np.nan)

print("✅ Valeurs -1 remplacées par NaN")
print(lieux_df[cols_sentinel].isna().sum())

✅ Valeurs -1 remplacées par NaN
catr         0
nbv       4737
circ     15773
prof       280
surf       269
infra     3048
situ       251
vma       9980
dtype: int64


In [4]:
# ── Correction du type de la colonne nbv ───────────────────────
# nbv (nombre de voies) est en object à cause de valeurs mixtes entre années
# On convertit en float (pas int car il peut y avoir des NaN)
# errors='coerce' transforme les valeurs non convertibles en NaN

lieux_df['nbv'] = pd.to_numeric(lieux_df['nbv'], errors='coerce')
print(f"✅ nbv converti en float")
print(f"Type nbv : {lieux_df['nbv'].dtype}")
print(f"NaN nbv  : {lieux_df['nbv'].isna().sum()}")

✅ nbv converti en float
Type nbv : float64
NaN nbv  : 4842


In [5]:
# ── Renommage colonnes lieux ─────────────────────────────────────
lieux_df = lieux_df.rename(columns={
    'catr'  : 'categorie_route',
    'circ'  : 'regime_circulation',
    'nbv'   : 'nb_voies',
    'prof'  : 'profil_route',
    'surf'  : 'etat_surface',
    'infra' : 'infrastructure',
    'situ'  : 'situation_accident',
    'vma'   : 'vitesse_max_autorisee',
})
print(f"✅ Colonnes renommées : {list(lieux_df.columns)}")

✅ Colonnes renommées : ['Num_Acc', 'categorie_route', 'regime_circulation', 'nb_voies', 'profil_route', 'etat_surface', 'infrastructure', 'situation_accident', 'vitesse_max_autorisee', 'annee']


In [6]:
# ── Vérification finale avant export ────────────────────────────

print(f"Shape final : {lieux_df.shape}")

print("\n--- Types des colonnes ---")
print(lieux_df.dtypes)

print("\n--- NaN par colonne ---")
nan_df = pd.DataFrame({
    'NaN count': lieux_df.isna().sum(),
    'NaN %': (lieux_df.isna().mean() * 100).round(1)
})
print(nan_df[nan_df['NaN count'] > 0])

print("\n--- Vérif aucun -1 restant ---")
for col in lieux_df.select_dtypes(include='number').columns:
    n = (lieux_df[col] == -1).sum()
    if n > 0:
        print(f"  ⚠️  {col} : {n} valeurs -1 restantes")
print("✅ Aucun -1 restant")

print("\n--- Doublons ---")
print(f"Doublons Num_Acc : {lieux_df.duplicated('Num_Acc').sum()}")

Shape final : (252928, 10)

--- Types des colonnes ---
Num_Acc                   object
categorie_route            int64
regime_circulation       float64
nb_voies                 float64
profil_route             float64
etat_surface             float64
infrastructure           float64
situation_accident       float64
vitesse_max_autorisee    float64
annee                      int64
dtype: object

--- NaN par colonne ---
                       NaN count  NaN %
regime_circulation         15773    6.2
nb_voies                    4842    1.9
profil_route                 280    0.1
etat_surface                 269    0.1
infrastructure              3048    1.2
situation_accident           251    0.1
vitesse_max_autorisee       9980    3.9

--- Vérif aucun -1 restant ---
  ⚠️  nb_voies : 4714 valeurs -1 restantes
✅ Aucun -1 restant

--- Doublons ---
Doublons Num_Acc : 31884


In [7]:
# ── Export du fichier nettoyé ───────────────────────────────────
# Export avec séparateur point-virgule (;) cohérent avec les autres tables
# Ce fichier sera utilisé pour la jointure avec caract sur Num_Acc

lieux_df.to_csv('Dataset/lieux_2021_2024_clean.csv', index=False, sep=';', encoding='utf-8')
print(f"✅ Exporté — {len(lieux_df):,} lignes x {lieux_df.shape[1]} colonnes")

✅ Exporté — 252,928 lignes x 10 colonnes
